# 3. Multi-Depot Routing

## Business Context

**Eden Logistics** is a fictional last-mile delivery company serving **200 customers** across a large metropolitan area. As part of a **scenario planning exercise**, the company evaluates how different depot configurations impact routing efficiency, fleet utilisation, and delivery cost.

---

## Scenarios Overview

| # | Scenario                            | Description                                                                                                                                     | Depot Placement Strategy             | Key Question                                                       |
| - | ----------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------ | ------------------------------------------------------------------ |
| 1 | **Single Depot (Baseline)**         | All vehicles start and end at one central depot, corresponding to the C1_2_1 dataset used in `1-intro.ipynb`.                                   | Fixed (benchmark depot)              | What is the baseline routing cost?                                 |
| 2 | **Two-Depot Flexible Assignment**   | A second depot is introduced. cuOpt determines how many vehicles each depot uses and which customers they serve.                                | Original depot + K-Means (k=2)       | How does adding a depot improve network efficiency?                |
| 3 | **Three-Depot Flexible Assignment** | A third depot is added, further decentralising operations. cuOpt allocates vehicles dynamically across all three depots.                        | Original depot + K-Means (k=2)       | How does adding another depot improve network efficiency?          |
| 4 | **Depot Candidate Selection**       | K-Means (k=3) is run multiple times with different seeds to generate varied 3-depot configurations.                                                   | K-Means (k=3), multiple seeds        | Which depot locations yield the best performance?                  |

## Table of Contents
1. [Install Dependencies](#2-install-dependencies)
2. [Set Up Helper Functions](#3-set-up-helper-functions)
3. [Check API Endpoint](#4-check-api-endpoint)
4. [Dataset — C1_2_1 Benchmark](#5-dataset--c1_2_1-benchmark)
   - 5.1 [Download and Visualise](#51-download-and-visualise)
   - 5.2 [Two-Depot Setup via K-Means (k=2)](#52-two-depot-setup-via-k-means-k2)
   - 5.3 [Three-Depot Setup: Original Depot + K-Means (k=2)](#53-three-depot-setup-original-depot--k-means-k2)
   - 5.4 [Multiple K-Means Runs (k=3) for Scenario 4](#54-multiple-k-means-runs-k3-for-scenario-4)
   - 5.5 [vehicle_locations Explained](#55-vehicle_locations-explained)
5. [Scenario 1 — Single Depot (Baseline)](#6-scenario-1--single-depot-baseline)
   - 6.1 [Build Payload](#61-build-payload)
   - 6.2 [Solve and Visualise](#62-solve-and-visualise)
6. [Scenario 2 — 2-Depot Flexible Assignment](#7-scenario-2--2-depot-flexible-assignment)
   - 7.1 [Build Payload](#71-build-payload)
   - 7.2 [Solve and Visualise](#72-solve-and-visualise)
7. [Scenario 3 — 3-Depot Flexible Assignment](#8-scenario-3--3-depot-flexible-assignment)
   - 8.1 [Build Payload](#81-build-payload)
   - 8.2 [Solve and Visualise](#82-solve-and-visualise)
8. [Scenario 4 — Best 3-Depot Configuration from Multiple K-Means Runs](#9-scenario-4--best-3-depot-configuration-from-multiple-k-means-runs)
   - 9.1 [Build Per-Run Payloads](#91-build-per-run-payloads)
   - 9.2 [Solve All Runs and Analyse Results](#92-solve-all-runs-and-analyse-results)
   - 9.3 [Visualise Results](#93-visualise-results)
9. [Comparative Analysis](#10-comparative-analysis)
    - 10.1 [Results Summary Table](#101-results-summary-table)
    - 10.2 [Logistics Case Studies](#102-logistics-case-studies)
10. [Conclusion](#11-conclusion)
11. [Cleanup](#12-cleanup-recommended)

## 1. Install Dependencies


In [ ]:
# The following packages are required to run this notebook:
%pip install -q matplotlib scipy pandas requests scikit-learn

import copy
import os
import sys
import logging

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.cluster import KMeans

logging.basicConfig(level=logging.INFO)

## 2. Set Up Helper Functions
All utility functions are defined in `helper/utils.py` and imported below.

In [ ]:
sys.path.insert(0, './helper')

from utils import (
    create_from_file,
    build_multi_depot_payload,
    plot_instance,
    plot_multi_depot_routes,
    solve,
)

## 3. Check API Endpoint

Set `base_url` to the Vehicle Route Optimizer endpoint from your OCI stack Outputs.

In [ ]:
# Update this value to point to your own deployed cuOpt instance
base_url = "http://cuopt.YOUR-ENDPOINT.nip.io"

In [ ]:
health_url = f"{base_url}/cuopt/health"
try:
    response = requests.get(health_url)
    print(f"Health check URL : {health_url}")
    print(f"Status code      : {response.status_code}")
    print(f"Response         : {response.json()}")
except requests.RequestException as e:
    print(f"Health check failed: {e}")

## 4. Dataset — C1_2_1 Benchmark

### 4.1 Download and Visualise

We use the **Gehring & Homberger C1_2_1** instance: 200 customers with **clustered locations** and **tight time windows**. The depot sits near the centre of the coordinate grid.

| Instance | Distribution | Customers | Vehicles | Capacity | Best-known vehicles | Best-known cost |
|----------|--------------|-----------|----------|----------|--------------------|------------------|
| `C1_2_1` | Clustered   | 200       | 20       | 200      | 20                 | 2704.57          |

Best-known results are published by [SINTEF TOP](https://www.sintef.no/projectweb/top/vrptw/200-customers/).

In [ ]:
!mkdir -p ./cvrptw_data
!wget -q https://www.sintef.no/globalassets/project/top/vrptw/homberger/200/homberger_200_customer_instances.zip \
      -O ./cvrptw_data/homberger_data.zip
!unzip -q -o ./cvrptw_data/homberger_data.zip -d ./cvrptw_data/

homberger_c1_file = "./cvrptw_data/C1_2_1.TXT"
if not os.path.exists(homberger_c1_file):
    raise FileNotFoundError(
        f"Could not find {homberger_c1_file}. "
        "Please check that the download succeeded."
    )

orders_c1, vehicle_capacity_c1, n_vehicles_c1 = create_from_file(homberger_c1_file)
n_locations_c1 = len(orders_c1) - 1  # exclude depot at row 0

print('Number of customers          :', n_locations_c1)
print('Number of vehicles available :', n_vehicles_c1)
print('Capacity of each vehicle     :', vehicle_capacity_c1)
print('\nDepot coordinates            : '
      f"({orders_c1['xcord'].iloc[0]:.1f}, {orders_c1['ycord'].iloc[0]:.1f})")
print('\nFirst 5 rows:')
print(orders_c1.head(5).to_string(index=False))

In [ ]:
plot_instance(orders_c1, 'C1_2_1', 'Clustered')

### 4.2 Two-Depot Setup via K-Means (k=2)

To determine where to place the **second depot** for Scenario 2, we run **K-Means (k=2)** on the 200 customer locations. Depot 1 remains at the original benchmark location; **Depot 2** is placed at the K-Means centroid furthest from Depot 1.

In [ ]:
# ── K-Means clustering (k=2) to find the optimal Depot 2 location ────────────
customers_c1 = orders_c1.iloc[1:].reset_index(drop=True)  # exclude depot row

km2 = KMeans(n_clusters=2, random_state=42, n_init=1)
km2.fit(customers_c1[['xcord', 'ycord']])
centroids_2  = km2.cluster_centers_   
labels_2     = km2.labels_            

# Depot 2 = centroid that is furthest from the original Depot 1
depot1_pos  = np.array([orders_c1['xcord'].iloc[0], orders_c1['ycord'].iloc[0]])
dists_to_d1 = np.linalg.norm(centroids_2 - depot1_pos, axis=1)
d2_centroid_idx = np.argmax(dists_to_d1)   # index into centroids_2
depot2_x, depot2_y = centroids_2[d2_centroid_idx]

depot1_row = orders_c1.iloc[0]
depot2_row = pd.DataFrame([{
    'vertex'       : 201,
    'xcord'        : depot2_x,
    'ycord'        : depot2_y,
    'demand'       : 0,
    'earliest_time': int(depot1_row['earliest_time']),
    'latest_time'  : int(depot1_row['latest_time']),
    'service_time' : 0,
}])

# Extended location list: Depot 1 (0) + 200 customers (1–200) + Depot 2 (201)
orders_extended = pd.concat([orders_c1, depot2_row], ignore_index=True)
depot2_idx = len(orders_extended) - 1  # 201

print('K-Means (k=2) cluster centroids:')
for i, c in enumerate(centroids_2):
    n_cust = int(np.sum(labels_2 == i))
    print(f'  Centroid {i}: x={c[0]:.2f}, y={c[1]:.2f}  ({n_cust} customers)')
print()
print(f'Depot 1 (index   0): x={depot1_row["xcord"]:.2f}, y={depot1_row["ycord"]:.2f}')
print(f'Depot 2 (index {depot2_idx}): x={depot2_x:.2f}, y={depot2_y:.2f}  ← K-Means derived')
print(f'\nTotal locations in extended dataset: {len(orders_extended)} '
      f'(1 original depot + {n_locations_c1} customers + 1 K-Means depot)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

# Colour customers by K-Means cluster assignment
cluster_colors_2 = ['#4c72b0', '#dd8452']
for cluster_id in range(2):
    mask = (labels_2 == cluster_id)
    cluster_cust = customers_c1[mask]
    ax.scatter(
        cluster_cust['xcord'], cluster_cust['ycord'],
        s=15, c=cluster_colors_2[cluster_id], alpha=0.7, zorder=3,
        label=f'Cluster {cluster_id + 1} ({int(mask.sum())} customers)'
    )

# Depot 1 (original benchmark)
d1 = orders_extended.iloc[0]
ax.scatter(d1['xcord'], d1['ycord'], s=160, c='red', marker='*',
           zorder=6, label='Depot 1 (benchmark origin)')
ax.annotate('Depot 1\n(origin)', (d1['xcord'], d1['ycord']),
            textcoords='offset points', xytext=(8, 6),
            fontsize=9, fontweight='bold', color='red')

# Depot 2 (K-Means)
d2 = orders_extended.iloc[-1]
ax.scatter(d2['xcord'], d2['ycord'], s=80, c='black', marker='D',
           zorder=6, label='Depot 2 (K-Means, k=2)')
ax.annotate('Depot 2\n(K-Means)', (d2['xcord'], d2['ycord']),
            textcoords='offset points', xytext=(8, -16),
            fontsize=9, fontweight='bold', color='black')

ax.set_title(
    'C1_2_1: Two-Depot Layout Derived from K-Means Clustering (k=2)\n'
    'Customer colours show cluster membership; depot = cluster centroid',
    fontsize=12
)
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
plt.tight_layout()
plt.show()

### 5.3 Three-Depot Setup: Original Depot + K-Means (k=2)

For **Scenario 3**, we retain the original benchmark depot as **Depot A** and run **K-Means (k=2)** on the 200 customer locations to derive two additional depots — **Depot B** and **Depot C**.

This approach preserves the existing infrastructure (Depot A) while using clustering to identify two natural customer groupings, positioning each new depot at the centroid of its cluster.

We provide more vehicles than needed at all three depots, allowing cuOpt to activate only the routes that reduce total cost — effectively balancing the fleet across depots.

In [ ]:
# ── K-Means clustering (k=2) to find two additional depot locations for S3 ────
km_s3 = KMeans(n_clusters=2, random_state=42, n_init=1)
km_s3.fit(customers_c1[['xcord', 'ycord']])
centroids_3  = km_s3.cluster_centers_   # shape (2, 2)
labels_3     = km_s3.labels_            # cluster assignment for each customer

# Sort centroids by x + y so Depot B is south-west of Depot C
sort_order   = np.argsort(centroids_3[:, 0] + centroids_3[:, 1])
centroids_3s = centroids_3[sort_order]
label_remap  = {old: new for new, old in enumerate(sort_order)}
labels_3s    = np.array([label_remap[l] for l in labels_3])

cluster_sizes_3 = [int(np.sum(labels_3s == i)) for i in range(2)]

ORD_B = ord('B')   # 66 — depot naming starts at B (Depot A is the original)

print('K-Means (k=2) centroids for Depots B and C (sorted SW → NE):')
for i, (c, sz) in enumerate(zip(centroids_3s, cluster_sizes_3)):
    print(f'  Depot {chr(ORD_B+i)}: x={c[0]:.2f}, y={c[1]:.2f}  ({sz} customers)')

# ── Build orders_3depot ───────────────────────────────────────────────────────
# Row 0      = Depot A (original benchmark depot, unchanged)
# Rows 1-200 = same 200 customers as before
# Row 201    = Depot B (K-Means centroid, SW cluster)
# Row 202    = Depot C (K-Means centroid, NE cluster)
orders_3depot = orders_c1.copy()   # row 0 is left unchanged

extra_depots = []
for i, c in enumerate(centroids_3s):
    extra_depots.append({
        'vertex'       : 201 + i,
        'xcord'        : c[0],
        'ycord'        : c[1],
        'demand'       : 0,
        'earliest_time': int(depot1_row['earliest_time']),
        'latest_time'  : int(depot1_row['latest_time']),
        'service_time' : 0,
    })
orders_3depot = pd.concat(
    [orders_3depot, pd.DataFrame(extra_depots)], ignore_index=True
)

depot_a_idx, depot_b_idx_3d, depot_c_idx = 0, 201, 202

print(f'\nTotal rows in orders_3depot: {len(orders_3depot)} '
      f'(3 depots + {n_locations_c1} customers)')
print(f'  Depot A → index {depot_a_idx}  (original benchmark depot, '
      f'x={depot1_row["xcord"]:.2f}, y={depot1_row["ycord"]:.2f})')
print(f'  Depot B → index {depot_b_idx_3d}  (K-Means centroid, '
      f'x={centroids_3s[0][0]:.2f}, y={centroids_3s[0][1]:.2f})')
print(f'  Depot C → index {depot_c_idx}  (K-Means centroid, '
      f'x={centroids_3s[1][0]:.2f}, y={centroids_3s[1][1]:.2f})')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

# Colour customers by K-Means (k=2) cluster assignment — Cluster 0 → Depot B, Cluster 1 → Depot C
cluster_colors_3   = ['#4c72b0', '#d62728']
depot_label_colors = ['#2ca02c', '#4c72b0', '#d62728']   # green=A, blue=B, red=C
depot_markers_3    = ['s', 'D', '^']

for cluster_id in range(2):
    mask = (labels_3s == cluster_id)
    cluster_cust = customers_c1[mask]
    ax.scatter(
        cluster_cust['xcord'], cluster_cust['ycord'],
        s=15, c=cluster_colors_3[cluster_id], alpha=0.6, zorder=3,
        label=f'Cluster {cluster_id + 1} → Depot {chr(ORD_B + cluster_id)} ({int(mask.sum())} customers)'
    )

# Depot A: original benchmark depot
depot_a_row = orders_3depot.iloc[depot_a_idx]
ax.scatter(depot_a_row['xcord'], depot_a_row['ycord'],
           s=160, c=depot_label_colors[0], marker=depot_markers_3[0], zorder=7,
           label='Depot A (original benchmark depot)')
ax.annotate('Depot A', (depot_a_row['xcord'], depot_a_row['ycord']),
            textcoords='offset points', xytext=(8, 6),
            fontsize=9, fontweight='bold', color=depot_label_colors[0])

# Depots B and C: K-Means (k=2) centroids
for i, didx in enumerate([depot_b_idx_3d, depot_c_idx]):
    row = orders_3depot.iloc[didx]
    dx, dy = row['xcord'], row['ycord']
    depot_name = chr(ORD_B + i)   # B, C
    ax.scatter(dx, dy, s=80, c=depot_label_colors[i + 1],
               marker=depot_markers_3[i + 1], zorder=7,
               label=f'Depot {depot_name} (K-Means centroid, {cluster_sizes_3[i]} customers)')
    ax.annotate(f'Depot {depot_name}', (dx, dy),
                textcoords='offset points', xytext=(8, 6),
                fontsize=9, fontweight='bold', color=depot_label_colors[i + 1])

ax.set_title(
    'C1_2_1: Three-Depot Layout — Original Depot A + K-Means (k=2) Depots B & C\n'
    'Customer colours show K-Means (k=2) cluster membership',
    fontsize=12
)
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
plt.tight_layout()
plt.show()

### 5.4 Multiple K-Means Runs (k=3) for Scenario 4

For **Scenario 4**, K-Means (k=3) is run multiple times — each with a **different random seed** — to generate a set of distinct 3-depot configurations. cuOpt is solved independently for each, and results are compared to identify the **best-performing depot configuration**.

**How it works:**
1. K-Means (k=3) is executed `N_RUNS` times, each with a distinct `random_state` seed.
2. Each run produces three depot centroids (sorted SW → NE).
3. cuOpt is solved for each depot set using the same **flexible over-provisioning** as S3.
4. The configuration that minimises total cost is selected.

Extended location list for each Scenario 4 run:

```
Index  0        → Depot A  (K-Means centroid 1 for this run)
Index  1–200    → 200 customers
Index  201      → Depot B  (K-Means centroid 2 for this run)
Index  202      → Depot C  (K-Means centroid 3 for this run)
```

In [ ]:
# ── Run K-Means (k=3) with multiple random seeds to generate distinct depot sets ──
N_RUNS   = 6
SEEDS_S4 = [0, 99, 123, 42, 756, 186]   # distinct initialisations

km_s4_runs = []
for seed in SEEDS_S4:
    km = KMeans(n_clusters=3, random_state=seed, n_init=1)
    km.fit(customers_c1[['xcord', 'ycord']])
    centroids = km.cluster_centers_   # shape (3, 2)
    labels    = km.labels_

    # Sort centroids SW → NE for consistent Depot A / B / C labelling
    sort_order    = np.argsort(centroids[:, 0] + centroids[:, 1])
    centroids_s   = centroids[sort_order]
    label_remap   = {old: new for new, old in enumerate(sort_order)}
    labels_s      = np.array([label_remap[l] for l in labels])
    cluster_sizes = [int(np.sum(labels_s == i)) for i in range(3)]

    km_s4_runs.append({
        'seed'         : seed,
        'centroids'    : centroids_s,
        'labels'       : labels_s,
        'cluster_sizes': cluster_sizes,
    })

print(f'Generated {N_RUNS} K-Means (k=3) depot sets (seeds: {SEEDS_S4})\n')
for i, run in enumerate(km_s4_runs):
    print(f'  Run {i+1} (seed={run["seed"]})')
    for j, (c, sz) in enumerate(zip(run['centroids'], run['cluster_sizes'])):
        print(f'    Depot {chr(65+j)}: x={c[0]:.2f}, y={c[1]:.2f}  ({sz} customers)')
    print()


In [ ]:
import math

# ── Visualise all K-Means (k=3) runs for Scenario 4 ─────────────────────────
cluster_colors_3 = ['#4c72b0', '#2ca02c', '#d62728']
depot_markers_3  = ['*', 'D', '^']

ncols = 3
nrows = math.ceil(N_RUNS / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten() if N_RUNS > 1 else [axes]

for ax_idx, run in enumerate(km_s4_runs):
    ax = axes[ax_idx]
    for cluster_id in range(3):
        mask = (run['labels'] == cluster_id)
        cluster_cust = customers_c1[mask]
        ax.scatter(
            cluster_cust['xcord'], cluster_cust['ycord'],
            s=12, c=cluster_colors_3[cluster_id], alpha=0.5, zorder=3,
        )
    for j, c in enumerate(run['centroids']):
        ax.scatter(c[0], c[1], s=120, c=cluster_colors_3[j],
                   marker=depot_markers_3[j], zorder=7, edgecolors='black', linewidths=0.5)
        ax.annotate(f'D{chr(65+j)}', (c[0], c[1]),
                    textcoords='offset points', xytext=(6, 5),
                    fontsize=8, fontweight='bold', color=cluster_colors_3[j])
    ax.set_title(f'Run {ax_idx+1}  (seed={run["seed"]})', fontsize=10)
    ax.set_xlabel('X'); ax.set_ylabel('Y')

# Hide any unused subplots
for ax_idx in range(N_RUNS, len(axes)):
    axes[ax_idx].set_visible(False)

fig.suptitle(
    f'Scenario 4 — {N_RUNS} K-Means (k=3) Depot Sets\n'
    f'(★ = Depot A, ◆ = Depot B, △ = Depot C; each subplot is a distinct random seed)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()


### 5.5 `vehicle_locations` Explained

The `vehicle_locations` field in a cuOpt payload is a **list of `[start, end]` pairs** — one
pair per vehicle. Each element is an index into the cost matrix, so it can point to any location
in the problem.

#### Baseline (S1) — fixed single depot
```python
# All 20 vehicles start AND end at Depot 1 (index 0)
vehicle_locations = [[0, 0]] * 20
```

#### Flexible 2-depot assignment (S2) — cuOpt decides the split
```python
# 40 vehicle slots: 20 at Depot 1 (original), 20 at Depot 2 (K-Means k=2)
# cuOpt activates only the vehicles that reduce cost
vehicle_locations = [[0,   0]]   * 20 +  # Depot 1 candidates (original)
                    [[201, 201]] * 20     # Depot 2 candidates (K-Means k=2)
```

#### Flexible 3-depot assignment (S3) — original depot + two K-Means depots
```python
# 60 vehicle slots: 20 at Depot A (original), 20 at Depot B (K-Means k=2), 20 at Depot C (K-Means k=2)
vehicle_locations = [[0,   0]]   * 20 +  # Depot A candidates (original)
                    [[201, 201]] * 20 +  # Depot B candidates (K-Means k=2)
                    [[202, 202]] * 20    # Depot C candidates (K-Means k=2)
```

#### Multi-seed K-Means (S4) — run K-Means (k=3) with different seeds, solve each
```python
# For each seed, 60 vehicle slots: 20 at each of Depot A, B, C
# (all three depot positions are K-Means derived and vary per seed)
for seed in SEEDS_S4:
    vehicle_locations = [
        [idx, idx]
        for idx in [0, 201, 202]
        for _ in range(n_vehicles)
    ]
    # run cuOpt and record (cost, vehicles_used, seed)
```

> **Key insight**: Re-running K-Means with different seeds produces varied depot layouts.
> Solving cuOpt for each layout identifies which spatial depot configuration
> minimises total travel cost and fleet size.

## 6. Scenario 1 — Single Depot (Baseline)

In this baseline scenario every vehicle **departs from and returns to the same depot** —
Depot 1 at location index 0 (the standard CVRPTW assumption).

```python
vehicle_locations = [[0, 0]] * 20  # all vehicles: start=Depot 1, end=Depot 1
```

This is the classic **closed-route, single-depot** formulation. The solver's only task is to
find the shortest feasible delivery sequences.

We use the extended cost matrix (202×202, which includes the K-Means Depot 2 location) so that
results are directly comparable with Scenario 2. The extra depot row/column has no effect on
this run as no vehicle references it.

### 6.1 Build Payload

In [ ]:
# Solver time limit used for all scenarios (seconds)
TIME_LIMIT = 60

In [ ]:
# Scenario 1: every vehicle starts and ends at Depot 1 (index 0)
vehicle_locs_s1 = [[0, 0]] * n_vehicles_c1

payload_s1 = build_multi_depot_payload(
    orders_df        = orders_extended,
    n_locations      = n_locations_c1,
    vehicle_capacity = vehicle_capacity_c1,
    vehicle_locations= vehicle_locs_s1,
)

print('Payload keys          :', list(payload_s1.keys()))
print('Cost matrix size      :',
      len(payload_s1['cost_matrix_data']['data']['0']),
      '×',
      len(payload_s1['cost_matrix_data']['data']['0'][0]))
print('vehicle_locations[:3] :', payload_s1['fleet_data']['vehicle_locations'][:3])
print('All vehicles start and end at index 0 (Depot 1)')

### 6.2 Solve and Visualise

In [ ]:
print(f'Scenario 1 — Standard (time limit: {TIME_LIMIT}s)')
response_s1 = solve(base_url, payload_s1, TIME_LIMIT)

results = {}  # collect all three scenarios for comparison

if response_s1.get('status') == 0:
    n_veh_s1   = response_s1.get('num_vehicles')
    cost_s1    = response_s1.get('solution_cost')
    results[1] = {'label': 'Single Depot (Baseline)', 'vehicles': n_veh_s1, 'cost': cost_s1, 'depots': 1}

    print(f'Vehicles used : {n_veh_s1}')
    print(f'Solution cost : {cost_s1:.2f}')

    plot_multi_depot_routes(
        response_s1, orders_extended, vehicle_locs_s1,
        title='Scenario 1 — Standard: All Vehicles Return to Depot 1',
        max_routes=20, detail_routes=20,
    )
else:
    print(f"Optimisation failed: status={response_s1.get('status')}")

## 7. Scenario 2 — 2-Depot Flexible Assignment

Eden Logistics adds a second depot (Depot 2, K-Means derived) and asks:
**"How should vehicles be split between the two depots?"**

Rather than pre-assigning vehicles to depots, we provide cuOpt with **more vehicles than needed
at both depots** and let the solver determine the optimal allocation:

```python
# 2 × n_vehicles slots: n at Depot 1, n at Depot 2
# cuOpt activates only the vehicles that reduce cost
vehicle_locations = [[0,   0]]   * n_vehicles +   # Depot 1 slots
                    [[201, 201]] * n_vehicles      # Depot 2 slots
```

**Why this works:**
- Vehicles not dispatched carry zero cost in the objective function.
- The solver activates a vehicle only if assigning it to at least one customer improves the
  solution — effectively determining the depot-vehicle split automatically.
- The result reveals both *how many* vehicles each depot uses and *which customers* each serves.

**Depot 2 location:** derived from K-Means (k=2) in Section 5.2 — positioned at the centroid
of the customer cluster furthest from Depot 1.

### 7.1 Build Payload

In [ ]:
# Scenario 2: 2 depots with flexible assignment
# Provide n_vehicles slots at EACH depot — cuOpt decides how many to use per depot
vehicle_locs_s2 = (
    [[0,         0        ]] * n_vehicles_c1 +   # Depot 1 (index 0)  ← original depot
    [[depot2_idx, depot2_idx]] * n_vehicles_c1   # Depot 2 (index 201) ← K-Means derived
)

total_slots_s2 = len(vehicle_locs_s2)

payload_s2 = build_multi_depot_payload(
    orders_df        = orders_extended,
    n_locations      = n_locations_c1,
    vehicle_capacity = vehicle_capacity_c1,
    vehicle_locations= vehicle_locs_s2,
)

print(f'Depot 1 → index 0     (original benchmark depot)')
print(f'Depot 2 → index {depot2_idx}   (K-Means centroid, k=2)')
print(f'vehicle_locations[:3] : {payload_s2["fleet_data"]["vehicle_locations"][:3]}')
print(f'vehicle_locations[-3:]: {payload_s2["fleet_data"]["vehicle_locations"][-3:]}')

### 7.2 Solve and Visualise

In [ ]:
print(f'Scenario 2 — 2-Depot Flexible Assignment (time limit: {TIME_LIMIT}s)')
response_s2 = solve(base_url, payload_s2, TIME_LIMIT)

if response_s2.get('status') == 0:
    n_veh_s2   = response_s2.get('num_vehicles')
    cost_s2    = response_s2.get('solution_cost')
    results[2] = {'label': '2-Depot Flexible', 'vehicles': n_veh_s2, 'cost': cost_s2,
                  'depots': 2}

    # Count vehicles dispatched from each depot
    depot_veh_count_s2 = {}
    for vid, vdata in response_s2.get('vehicle_data', {}).items():
        route = vdata.get('route', [])
        if route:
            d = route[0]
            depot_veh_count_s2[d] = depot_veh_count_s2.get(d, 0) + 1

    used_depot_indices_s2 = set(depot_veh_count_s2.keys())

    print(f'Vehicles used          : {n_veh_s2}')
    print(f'Depot indices used     : {sorted(used_depot_indices_s2)}')
    print(f'Solution cost          : {cost_s2:.2f}')
    print('Depot utilisation:')
    for didx, cnt in sorted(depot_veh_count_s2.items()):
        depot_name = 'Depot 1 (original)' if didx == 0 else f'Depot 2 (K-Means, idx {didx})'
        print(f'  {depot_name}: {cnt} vehicle(s) dispatched')

    plot_multi_depot_routes(
        response_s2, orders_extended, vehicle_locs_s2,
        title='Scenario 2 — 2-Depot Flexible Assignment\ncuOpt determines which depot serves which customers',
        max_routes=20, detail_routes=20,
    )
else:
    print(f"Optimisation failed: status={response_s2.get('status')}")

## 8. Scenario 3 — 3-Depot Flexible Assignment

Eden Logistics now operates **three distribution centres** — Depot A (the original benchmark
depot), Depot B and Depot C (K-Means derived, k=2). The planning question becomes:

> *"Which depot should each vehicle be assigned to, and which customers should each depot serve?"*

As in Scenario 2, we solve this by **over-provisioning vehicles** at all three depots:

```python
# 3 × n_vehicles slots: n at each of Depot A, B, C
vehicle_locations = [[depot_A, depot_A]] * n_vehicles +   # Depot A slots
                    [[depot_B, depot_B]] * n_vehicles +   # Depot B slots
                    [[depot_C, depot_C]] * n_vehicles     # Depot C slots
```

**What this adds over Scenario 2:**
- Three geographic zones, each with its own depot.
- The solver simultaneously optimises the **depot-to-customer assignment** and the
  **delivery sequence** within each route — a more complex search space.
- The result shows how a three-hub network distributes demand across zones.

**Depot locations:**
- Depot A (index 0): original benchmark depot
- Depot B (index 201): K-Means (k=2) centroid, south-west cluster
- Depot C (index 202): K-Means (k=2) centroid, north-east cluster

### 8.1 Build Payload

In [ ]:
# Scenario 3: 3 depots with flexible assignment
# Provide n_vehicles slots at EACH depot — cuOpt decides how many to use per depot
vehicle_locs_s3 = (
    [[depot_a_idx,    depot_a_idx   ]] * n_vehicles_c1 +   # Depot A (index 0)
    [[depot_b_idx_3d, depot_b_idx_3d]] * n_vehicles_c1 +   # Depot B (index 201)
    [[depot_c_idx,    depot_c_idx   ]] * n_vehicles_c1     # Depot C (index 202)
)

total_slots_s3 = len(vehicle_locs_s3)

payload_s3 = build_multi_depot_payload(
    orders_df        = orders_3depot,
    n_locations      = n_locations_c1,
    vehicle_capacity = vehicle_capacity_c1,
    vehicle_locations= vehicle_locs_s3,
)

print(f'Depot A → index {depot_a_idx:3d}  (original benchmark depot)')
print(f'Depot B → index {depot_b_idx_3d:3d}  (K-Means centroid, k=2, SW cluster)')
print(f'Depot C → index {depot_c_idx:3d}  (K-Means centroid, k=2, NE cluster)')
print(f'vehicle_locations[:2]  : {payload_s3["fleet_data"]["vehicle_locations"][:2]}')
print(f'vehicle_locations[-2:] : {payload_s3["fleet_data"]["vehicle_locations"][-2:]}')
print('  → cuOpt will use only the vehicles that reduce total cost')

### 8.2 Solve and Visualise

In [ ]:
print(f'Scenario 3 — 3-Depot Flexible Assignment (time limit: {TIME_LIMIT}s)')
response_s3 = solve(base_url, payload_s3, TIME_LIMIT)

if response_s3.get('status') == 0:
    n_veh_s3   = response_s3.get('num_vehicles')
    cost_s3    = response_s3.get('solution_cost')
    results[3] = {'label': '3-Depot Flexible', 'vehicles': n_veh_s3, 'cost': cost_s3,
                  'depots': 3}

    # Count vehicles dispatched from each depot
    depot_veh_count_s3 = {}
    for vid, vdata in response_s3.get('vehicle_data', {}).items():
        route = vdata.get('route', [])
        if route:
            d = route[0]
            depot_veh_count_s3[d] = depot_veh_count_s3.get(d, 0) + 1

    depot_names_s3 = {depot_a_idx: 'Depot A', depot_b_idx_3d: 'Depot B', depot_c_idx: 'Depot C'}

    print(f'Vehicles used          : {n_veh_s3}')
    print(f'Solution cost          : {cost_s3:.2f}')
    print('Depot utilisation:')
    for didx in [depot_a_idx, depot_b_idx_3d, depot_c_idx]:
        cnt = depot_veh_count_s3.get(didx, 0)
        print(f'  {depot_names_s3[didx]} (index {didx:3d}): {cnt} vehicle(s) dispatched')

    plot_multi_depot_routes(
        response_s3, orders_3depot, vehicle_locs_s3,
        title='Scenario 3 — 3-Depot Flexible Assignment\ncuOpt determines which depot serves which customers',
        max_routes=20, detail_routes=20,
    )
else:
    print(f"Optimisation failed: status={response_s3.get('status')}")

---

## Flexible Assignment Recap: Scenarios 1–3

The first three scenarios illustrate the progression from a fixed single-depot formulation to
a fully flexible three-depot assignment:

| Scenario | Vehicle slots | Depot locations | Solver freedom |
|----------|--------------|-----------------|----------------|
| S1 — Baseline | `n` (1 × n) | 1 (original) | Route order only |
| S2 — 2-Depot Flexible | `2n` (n per depot) | 2 (original + K-Means k=2) | Split + route order |
| S3 — 3-Depot Flexible | `3n` (n per depot) | 3 (original + K-Means k=2) | Split across 3 + route order |

In S2 and S3, the solver determines **both** the depot assignment **and** the routing. The depot
locations are fixed — the only unknowns are *how many vehicles* each depot uses and *which
customers* each vehicle serves.

**Scenario 4 adds one more layer**: the depot *locations themselves* become a variable — by
re-running K-Means (k=3) with different random seeds, each producing a distinct 3-depot
configuration. cuOpt is solved for every configuration, and the best-performing set is selected.

---

## 9. Scenario 4 — Best 3-Depot Configuration from Multiple K-Means Runs

Scenario 4 explores **whether the choice of K-Means random seed for k=3 affects the
optimisation outcome**. By running K-Means (k=3) `N_RUNS` times — each with a different
random initialisation — we obtain a set of distinct 3-depot configurations. cuOpt is solved
independently for each using the same **flexible over-provisioning** strategy as S3.

After all `N_RUNS` solves, results are analysed to find the configuration that minimises:
1. **Total travel cost** (primary metric).
2. **Vehicles used** (secondary metric, tie-break).

The winning configuration is then visualised together with its routes.

### 9.1 Build Per-Run Payloads

In [ ]:
# ── Build a separate orders DataFrame and vehicle-location list for each run ──
s4_run_payloads = []

for i, run in enumerate(km_s4_runs):
    centroids_s = run['centroids']   # shape (3, 2), sorted SW → NE

    # Depot A replaces row 0; Depots B and C are appended after the customers
    orders_run = orders_c1.copy()
    orders_run.at[0, 'xcord'] = centroids_s[0][0]
    orders_run.at[0, 'ycord'] = centroids_s[0][1]

    extra_rows = []
    for k, c in enumerate(centroids_s[1:], start=1):
        extra_rows.append({
            'vertex'       : 200 + k,
            'xcord'        : c[0],
            'ycord'        : c[1],
            'demand'       : 0,
            'earliest_time': int(depot1_row['earliest_time']),
            'latest_time'  : int(depot1_row['latest_time']),
            'service_time' : 0,
        })
    orders_run = pd.concat(
        [orders_run, pd.DataFrame(extra_rows)], ignore_index=True
    )

    # Flexible assignment: n_vehicles slots at each of the 3 depots
    depot_indices_run = [0, 201, 202]
    vehicle_locs = [
        [idx, idx]
        for idx in depot_indices_run
        for _ in range(n_vehicles_c1)
    ]

    payload = build_multi_depot_payload(
        orders_df        = orders_run,
        n_locations      = n_locations_c1,
        vehicle_capacity = vehicle_capacity_c1,
        vehicle_locations= vehicle_locs,
    )

    s4_run_payloads.append({
        'run_idx'     : i,
        'seed'        : run['seed'],
        'centroids'   : centroids_s,
        'orders_df'   : orders_run,
        'vehicle_locs': vehicle_locs,
        'payload'     : payload,
    })

N_RUNS_PAYLOADS = len(s4_run_payloads)
print(f'Built {N_RUNS_PAYLOADS} payloads for Scenario 4 (one per K-Means run)')
print()
for p in s4_run_payloads:
    cents = p['centroids']
    print(f'  Run {p["run_idx"]+1} (seed={p["seed"]}): '
          f'A=({cents[0][0]:.1f},{cents[0][1]:.1f})  '
          f'B=({cents[1][0]:.1f},{cents[1][1]:.1f})  '
          f'C=({cents[2][0]:.1f},{cents[2][1]:.1f})')


### 9.2 Solve All Runs and Analyse Results

In [ ]:
import math

run_results_s4 = []   # list of dicts, one per K-Means run

print(f'Scenario 4 — Evaluating {N_RUNS_PAYLOADS} K-Means (k=3) depot configurations '
      f'(time limit: {TIME_LIMIT}s each)\n')

for p in s4_run_payloads:
    run_label = f'Run {p["run_idx"]+1} (seed={p["seed"]})'

    print(f'  [{p["run_idx"]+1}/{N_RUNS_PAYLOADS}] Solving {run_label} ...', end=' ', flush=True)
    resp = solve(base_url, p['payload'], TIME_LIMIT)

    if resp.get('status') == 0:
        cost     = resp.get('solution_cost')
        n_veh    = resp.get('num_vehicles')
        n_served = sum(
            len(vd.get('route', [])) - 2
            for vd in resp.get('vehicle_data', {}).values()
            if len(vd.get('route', [])) >= 2
        )
        print(f'cost={cost:.2f}  vehicles={n_veh}  served={n_served}')
    else:
        cost, n_veh, n_served = math.inf, None, 0
        print(f'FAILED (status={resp.get("status")})')

    run_results_s4.append({
        'run_idx'     : p['run_idx'],
        'seed'        : p['seed'],
        'label'       : run_label,
        'centroids'   : p['centroids'],
        'vehicle_locs': p['vehicle_locs'],
        'orders_df'   : p['orders_df'],
        'response'    : resp,
        'cost'        : cost,
        'vehicles'    : n_veh,
        'served'      : n_served,
    })

# ── Identify best run (lowest cost, fewest vehicles as tie-break) ─────────────
valid_runs = [r for r in run_results_s4 if r['cost'] < math.inf]
if valid_runs:
    best_s4  = min(valid_runs, key=lambda r: (r['cost'], r['vehicles'] or math.inf))
    worst_s4 = max(valid_runs, key=lambda r: r['cost'])

    print()
    print('=' * 70)
    print(f'S4 Results — {N_RUNS_PAYLOADS} K-Means (k=3) Depot Configurations')
    print('=' * 70)

    sorted_runs = sorted(valid_runs, key=lambda r: r['cost'])
    print(f'  {"Rank":<5} {"Configuration":<28} {"Cost":>10} {"Vehicles":>9} {"Served":>7}')
    print(f'  {"-"*5} {"-"*28} {"-"*10} {"-"*9} {"-"*7}')
    for rank, r in enumerate(sorted_runs, 1):
        marker = ' ← BEST' if r is best_s4 else ''
        print(f'  {rank:<5} {r["label"]:<28} {r["cost"]:>10.2f} '
              f'{str(r["vehicles"]):>9} {r["served"]:>7}{marker}')

    print()
    print(f'Best configuration : {best_s4["label"]}')
    cents = best_s4['centroids']
    for j, c in enumerate(cents):
        print(f'  Depot {chr(65+j)}: x={c[0]:.2f}, y={c[1]:.2f}')
    print(f'  Cost             : {best_s4["cost"]:.2f}')
    print(f'  Vehicles used    : {best_s4["vehicles"]}')
    print(f'  Customers served : {best_s4["served"]}')
    spread = worst_s4['cost'] - best_s4['cost']
    print(f'  Cost range       : {spread:.2f} '
          f'({spread / best_s4["cost"] * 100:.1f}% spread across all runs)')

    # Store best S4 result for comparative analysis
    results[4] = {
        'label'   : f'3-Depot Optimised',
        'vehicles': best_s4['vehicles'],
        'cost'    : best_s4['cost'],
        'depots'  : 3,
        'slots'   : len(best_s4['vehicle_locs']),
    }
else:
    print('\nAll S4 runs failed — check the API endpoint.')
    best_s4 = None


### 9.3 Visualise Results

#### Run Cost Comparison

In [ ]:
# ── Bar chart: cost across all K-Means runs ──────────────────────────────────
if valid_runs:
    sorted_runs_plot  = sorted(valid_runs, key=lambda r: r['cost'])
    run_labels_plot   = [r['label']  for r in sorted_runs_plot]
    costs_plot        = [r['cost']   for r in sorted_runs_plot]
    colors_bar        = ['#2ecc71' if r is best_s4 else '#4c72b0' for r in sorted_runs_plot]

    fig, ax = plt.subplots(figsize=(max(10, 2 * len(sorted_runs_plot)), 5))
    bars = ax.bar(range(len(run_labels_plot)), costs_plot, color=colors_bar,
                  edgecolor='white', alpha=0.9)
    ax.set_xticks(range(len(run_labels_plot)))
    ax.set_xticklabels(run_labels_plot, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Total Cost (travel distance)')
    ax.set_title(
        f'Scenario 4 — Cost for All {N_RUNS_PAYLOADS} K-Means (k=3) Depot Configurations\n'
        f'(green = best configuration: {best_s4["label"]})',
        fontsize=12
    )
    for bar, val in zip(bars, costs_plot):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8)

    # Annotate best run
    best_idx = sorted_runs_plot.index(best_s4)
    ax.annotate(
        'BEST', xy=(best_idx, best_s4['cost']),
        xytext=(best_idx, best_s4['cost'] + max(costs_plot) * 0.04),
        ha='center', fontsize=10, fontweight='bold', color='#27ae60',
        arrowprops=dict(arrowstyle='->', color='#27ae60', lw=1.5),
    )
    plt.tight_layout()
    plt.show()

# ── Route visualisation for the best K-Means run ─────────────────────────────
if best_s4 and best_s4['response'].get('status') == 0:
    cents = best_s4['centroids']
    depot_str = '  |  '.join(
        [f'Depot {chr(65+j)} ({c[0]:.1f}, {c[1]:.1f})' for j, c in enumerate(cents)]
    )
    plot_multi_depot_routes(
        best_s4['response'],
        best_s4['orders_df'],
        best_s4['vehicle_locs'],
        title=(
            f'Scenario 4 — Best K-Means Run: {best_s4["label"]}\n'
            f'{depot_str}\n'
            f'Cost: {best_s4["cost"]:.2f}  |  Vehicles: {best_s4["vehicles"]}'
        ),
        max_routes=20,
        detail_routes=20,
    )


## 10. Comparative Analysis

### 10.1 Results Summary Table

In [ ]:
if results:
    baseline_cost = results[1]['cost'] if 1 in results else None

    rows = []
    for s_num in sorted(results):
        r = results[s_num]
        cost = r['cost']

        delta = (
            (cost - baseline_cost) / baseline_cost * 100
            if (baseline_cost and s_num != 1)
            else None
        )

        rows.append({
            'Scenario'      : f"S{s_num}: {r['label']}",
            'Depots'        : r.get('depots', 'N/A'),
            'Vehicles Used' : r['vehicles'],
            'Total Cost'    : f"{cost:.2f}",
            'Δ vs Baseline' : f"{delta:+.2f}%" if delta is not None else '—',
        })

    df_summary = pd.DataFrame(rows).set_index('Scenario')

    print('=' * 80)
    print('cuOpt Multi-Depot Comparison — C1_2_1 (200 customers, flexible assignment)')
    print('=' * 80)

    try:
        from IPython.display import display
        display(df_summary)
    except ImportError:
        print(df_summary.to_string())
else:
    print('No results collected — check that all scenarios ran successfully.')

In [ ]:
if len(results) >= 2:
    sorted_items = sorted(results.items())
    labels = [f"S{k}: {v['label']}" for k, v in sorted_items]
    costs  = [v['cost']     for _, v in sorted_items]
    n_vehs = [v['vehicles'] for _, v in sorted_items]

    bar_colors = ['#4c72b0', '#dd8452', '#55a868', '#c44e52']
    x = np.arange(len(labels))
    width = 0.5

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    bars1 = ax1.bar(x, costs, width, color=bar_colors[:len(labels)],
                    edgecolor='white', alpha=0.9)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=8, rotation=10, ha='right')
    ax1.set_ylabel('Total Cost (travel distance)')
    ax1.set_title('Total Travel Cost by Scenario')
    for bar, val in zip(bars1, costs):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=8)

    bars2 = ax2.bar(x, n_vehs, width, color=bar_colors[:len(labels)],
                    edgecolor='white', alpha=0.9)
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels, fontsize=8, rotation=10, ha='right')
    ax2.set_ylabel('Number of Vehicles Used')
    ax2.set_title('Vehicles Used by Scenario')
    for bar, val in zip(bars2, n_vehs):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 str(val), ha='center', va='bottom', fontsize=8)

    plt.suptitle(
        'Multi-Depot Comparison — C1_2_1 Benchmark (200 customers)',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()


## 10. Conclusion

This notebook demonstrated how cuOpt's `vehicle_locations` field enables a progressive
escalation of routing complexity — from a single-depot baseline to a solver-driven depot
location selection — all on the Gehring & Homberger C1_2_1 200-customer benchmark.

| Scenario | Configuration | Depot placement | Solver decides |
|----------|--------------|-----------------|----------------|
| **S1 — Single Depot** | `[D1, D1] × n` | Original benchmark depot | Route order only |
| **S2 — 2-Depot Flexible** | `[D1, D1] × n` + `[D2, D2] × n` | Original + K-Means (k=2) | Fleet split + routing |
| **S3 — 3-Depot Flexible** | `[Dₓ, Dₓ] × n` for each of 3 depots | Original + K-Means (k=2) | 3-way split + routing |
| **S4 — Best 3-Depot Config** | `[cᵢ, cᵢ] × n` for each of 3 depots | K-Means (k=3), multiple seeds | Best configuration + routing |

## 11. Cleanup (Recommended)

To avoid ongoing GPU/OKE costs, run **Destroy** on the Resource Manager stack in the OCI Console after you finish testing.

In [ ]:
# Optional: remove downloaded benchmark data
import shutil
if os.path.exists('./cvrptw_data'):
    shutil.rmtree('./cvrptw_data')
    print('Removed ./cvrptw_data')